# 1.4 完全不讀前文，能用什麼猜法？


先準備一袋下一字候選：三個「貓」、一個「狗」、一個「鳥」。如果完全不知道前文，至少能讓常見字取得較多機會：貓佔五份中的三份，機率是 3/5=0.6；狗與鳥各 0.2。

這種只數單字的方法叫 **unigram**。它提供一個容易理解的**基準**，也就是之後拿來比較的簡單起點。次數回答「出現幾次」，機率回答「佔全部的多少比例」；三個候選的機率加總應是 1。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from collections import Counter

text = "貓貓貓狗鳥"
counts = Counter(text)
total = sum(counts.values())
probability = {}
for char, count in counts.items():
    probability[char] = count / total
print(counts)
print(probability)
assert abs(sum(probability.values()) - 1) < 1e-6

`Counter` 替每種字計數，第一行會印貓 3、狗 1、鳥 1。`sum(counts.values())` 得到總數 5；迴圈把各次數除以 5，第二行得到 `{'貓':0.6,'狗':0.2,'鳥':0.2}`。最後確認加總接近 1。

這張分配沒有收到前文。因此問「貓看」後面接什麼，或問「狗吃」後面接什麼，它都給貓 0.6。猜貓可能經常碰巧正確，卻不能說抓住了前後字的關係。

拿到分配後還要選字：永遠取最高者，本例每次都是貓；按機率抽選，狗與鳥仍有機會。計算分配與使用分配是兩步，這裡先完成第一步。

練習把材料換成「貓貓貓狗狗狗鳥」。貓的次數仍是 3，但總數變 7，機率降至 3/7≈0.429。先手算，再核對程式，分清次數沒有減少與佔比下降。

<details>
<summary>補充與重做</summary>

`abs` 取絕對值，`1e-6` 是 0.000001。電腦儲存小數有微小誤差，因此用這個容忍範圍檢查總和，並非允許漏掉候選。機率見 [W.4](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.4)，字典迴圈見 [W.2](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.2)。

</details>
